<a href="https://colab.research.google.com/github/JuanCappelletti/7-Preprocesamiento/blob/master/01_Consolidar_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***COMPILADOR DE TOMAS DE PHINDOWS EXPORTADOR***

In [ ]:
# @title

# ============================================================
# ANALISIS COMPLETO DE CAUDAL - GOOGLE COLAB
# VERSION: PROMEDIOS REALES + HEATMAP CORREGIDO
#
# MODIFICACIONES:
# - El HTML NO incluye detalle de registros faltantes
# - El Excel SI conserva detalle de faltantes
# - Los faltantes NO participan en los promedios
# - Los ceros realmente medidos SI participan
# - Heatmap con cero blanco
# - Cursor con valor correcto
# - Sin grafico de duracion bajo limite
#
# COPIAR TODO EN UNA SOLA CELDA DE GOOGLE COLAB
# ============================================================

!pip install -q plotly openpyxl

import io
import re
import html
import calendar
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from plotly.offline import plot
from google.colab import files
from IPython.display import display, HTML
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# CONFIGURACION
# ============================================================

LIMITE_ANOMALIA = 2200
INTERVALO_MINUTOS = 20
INTERVALO_HORAS = INTERVALO_MINUTOS / 60

# MDY = MES/DIA/ANIO
# DMY = DIA/MES/ANIO
FORMATO_FECHA_AMBIGUA = "MDY"

MOSTRAR_MESES_COMPLETOS = False
ELIMINAR_DUPLICADOS = True

MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo",
    4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre",
    10: "Octubre", 11: "Noviembre",
    12: "Diciembre"
}

# ============================================================
# MENSAJES DESTACADOS
# ============================================================

def cartel(titulo, mensaje, color="#FFF3CD"):

    display(HTML(f"""
    <div style="
        background:{color};
        border:3px solid #E5A000;
        border-radius:14px;
        padding:24px;
        margin:18px 0;
        text-align:center;
        font-family:Arial;
    ">
        <div style="
            font-size:27px;
            font-weight:bold;
            color:#754800;
            margin-bottom:12px;
        ">{html.escape(str(titulo))}</div>

        <div style="
            font-size:21px;
            line-height:1.5;
        ">{mensaje}</div>
    </div>
    """))

# ============================================================
# FUNCION PARA INTERPRETAR FECHAS
# ============================================================

def convertir_fechas(serie):

    s = serie.astype("string").str.strip()

    resultado = pd.Series(
        pd.NaT,
        index=serie.index,
        dtype="datetime64[ns]"
    )

    # ISO: AAAA-MM-DD
    mascara_iso = s.str.match(
        r"^\d{4}-\d{1,2}-\d{1,2}$",
        na=False
    )

    resultado.loc[mascara_iso] = pd.to_datetime(
        s.loc[mascara_iso],
        format="%Y-%m-%d",
        errors="coerce"
    )

    # Fechas con barras
    mascara_barras = s.str.match(
        r"^\d{1,2}/\d{1,2}/\d{4}$",
        na=False
    )

    partes = s.loc[mascara_barras].str.split(
        "/",
        expand=True
    )

    if len(partes):

        a = pd.to_numeric(
            partes[0],
            errors="coerce"
        )

        b = pd.to_numeric(
            partes[1],
            errors="coerce"
        )

        mascara_dmy = a > 12
        mascara_mdy = b > 12

        ambiguas = ~(mascara_dmy | mascara_mdy)

        if FORMATO_FECHA_AMBIGUA == "DMY":
            mascara_dmy |= ambiguas
        else:
            mascara_mdy |= ambiguas

        idx_dmy = partes.index[mascara_dmy]
        idx_mdy = partes.index[mascara_mdy]

        if len(idx_dmy):
            resultado.loc[idx_dmy] = pd.to_datetime(
                s.loc[idx_dmy],
                format="%d/%m/%Y",
                errors="coerce"
            )

        if len(idx_mdy):
            resultado.loc[idx_mdy] = pd.to_datetime(
                s.loc[idx_mdy],
                format="%m/%d/%Y",
                errors="coerce"
            )

    # Otros formatos, incluidas fechas con hora
    pendientes = resultado.isna() & s.notna()

    if pendientes.any():

        resultado.loc[pendientes] = pd.to_datetime(
            s.loc[pendientes],
            format="mixed",
            dayfirst=(FORMATO_FECHA_AMBIGUA == "DMY"),
            errors="coerce"
        )

    return resultado

# ============================================================
# FUNCION PARA CONVERTIR CAUDALES
# ============================================================

def convertir_numeros(serie):

    s = serie.astype("string").str.strip()
    s = s.str.replace(" ", "", regex=False)

    # Formato 1.234,56
    europeo = s.str.match(
        r"^-?\d{1,3}(\.\d{3})+,\d+$",
        na=False
    )

    s.loc[europeo] = (
        s.loc[europeo]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    # Formato 1234,56
    decimal_coma = (
        s.str.contains(",", regex=False, na=False)
        &
        ~s.str.contains(".", regex=False, na=False)
    )

    s.loc[decimal_coma] = (
        s.loc[decimal_coma]
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(s, errors="coerce")

# ============================================================
# FUNCION PARA INTERPRETAR HORAS
# ============================================================

def convertir_horas(serie):

    s = serie.astype("string").str.strip()

    # Acepta HH:MM y HH:MM:SS
    horas = pd.to_timedelta(
        s,
        errors="coerce"
    )

    # Valores horarios numericos de Excel
    pendientes = horas.isna()

    if pendientes.any():

        numericos = convertir_numeros(
            s.loc[pendientes]
        )

        # Excel representa la hora como fraccion del dia
        excel = numericos.between(
            0, 1, inclusive="left"
        )

        indices = numericos.index[excel]

        horas.loc[indices] = pd.to_timedelta(
            numericos.loc[indices],
            unit="D"
        )

    return horas

# ============================================================
# PASO 1 - CARGAR ARCHIVOS
# ============================================================

cartel(
    "PASO 1 - CARGAR ARCHIVOS",
    """
    Seleccioná todos los archivos CSV o Excel
    que contienen las mediciones de caudal.
    <br><br>
    <b>FECHA | HORA | CAUDAL</b>
    <br><br>
    Podés seleccionar varios archivos juntos.
    """
)

uploaded = files.upload()

if not uploaded:
    raise ValueError("No se cargaron archivos.")

dfs = []
control_archivos = []

# ============================================================
# LEER ARCHIVOS
# ============================================================

for nombre, contenido in uploaded.items():

    print("\nProcesando:", nombre)

    try:

        if nombre.lower().endswith(".csv"):

            try:
                temporal = pd.read_csv(
                    io.BytesIO(contenido),
                    header=None,
                    dtype=str,
                    sep=None,
                    engine="python",
                    encoding="utf-8-sig"
                )

            except UnicodeDecodeError:
                temporal = pd.read_csv(
                    io.BytesIO(contenido),
                    header=None,
                    dtype=str,
                    sep=None,
                    engine="python",
                    encoding="latin-1"
                )

        elif nombre.lower().endswith((".xlsx", ".xls")):

            temporal = pd.read_excel(
                io.BytesIO(contenido),
                header=None,
                dtype=str
            )

        else:
            print("Formato no admitido. Se omite.")
            continue

        if temporal.shape[1] < 3:
            print("ERROR: el archivo tiene menos de 3 columnas.")
            continue

        temporal = temporal.iloc[:, :3].copy()

        temporal.columns = [
            "Fecha",
            "Hora",
            "Caudal"
        ]

        cantidad_original = len(temporal)

        fechas = convertir_fechas(
            temporal["Fecha"]
        )

        horas = convertir_horas(
            temporal["Hora"]
        )

        # Si la fecha ya incluye hora, se conserva.
        # Si es solo fecha, se suma la columna Hora.
        fecha_base = fechas.dt.normalize()

        fecha_hora = fecha_base + horas

        tiene_hora_en_fecha = (
            fechas.notna()
            &
            (fechas != fecha_base)
        )

        fecha_hora.loc[tiene_hora_en_fecha] = (
            fechas.loc[tiene_hora_en_fecha]
        )

        temporal["FechaHora"] = fecha_hora

        temporal["Caudal"] = convertir_numeros(
            temporal["Caudal"]
        )

        invalidos_fecha = int(
            temporal["FechaHora"].isna().sum()
        )

        invalidos_caudal = int(
            temporal["Caudal"].isna().sum()
        )

        temporal = temporal.dropna(
            subset=["FechaHora", "Caudal"]
        ).copy()

        validos = len(temporal)

        control_archivos.append({
            "Archivo": nombre,
            "Registros originales": cantidad_original,
            "Registros validos": validos,
            "Descartados": cantidad_original - validos,
            "Fechas invalidas": invalidos_fecha,
            "Caudales invalidos": invalidos_caudal,
            "Primera fecha": (
                temporal["FechaHora"].min()
                if validos else pd.NaT
            ),
            "Ultima fecha": (
                temporal["FechaHora"].max()
                if validos else pd.NaT
            )
        })

        if validos:
            dfs.append(
                temporal[["FechaHora", "Caudal"]]
            )

        print("Registros originales:", cantidad_original)
        print("Registros válidos:", validos)
        print(
            "Registros descartados:",
            cantidad_original - validos
        )

    except Exception as error:

        print("ERROR:", nombre, error)

if not dfs:

    raise ValueError(
        "No se encontraron registros válidos."
    )

# ============================================================
# UNIFICAR ARCHIVOS
# ============================================================

df = pd.concat(
    dfs,
    ignore_index=True
)

df = df.sort_values("FechaHora")

duplicados = 0

if ELIMINAR_DUPLICADOS:

    duplicados = int(
        df.duplicated(
            subset=["FechaHora", "Caudal"]
        ).sum()
    )

    df = df.drop_duplicates(
        subset=["FechaHora", "Caudal"]
    )

df = df.sort_values("FechaHora").copy()

fecha_inicio = df["FechaHora"].min()
fecha_fin = df["FechaHora"].max()

inicio = fecha_inicio.normalize()
fin = fecha_fin.normalize()

if MOSTRAR_MESES_COMPLETOS:

    inicio = inicio.replace(day=1)

    fin = fin.replace(
        day=calendar.monthrange(
            fin.year,
            fin.month
        )[1]
    )

todos_los_dias = pd.date_range(
    inicio,
    fin,
    freq="D"
)

# ============================================================
# REMUESTREO DE 20 MINUTOS
# ============================================================

df = df.set_index(
    "FechaHora"
).sort_index()

# Cada intervalo tiene el promedio de las
# mediciones reales existentes dentro de el.

df20 = (
    df["Caudal"]
    .resample(f"{INTERVALO_MINUTOS}min")
    .mean()
    .reset_index()
)

df20["Dia"] = (
    df20["FechaHora"]
    .dt.strftime("%Y-%m-%d")
)

df20["Hora"] = (
    df20["FechaHora"]
    .dt.strftime("%H:%M")
)

# ============================================================
# MATRIZ COMPLETA DE DIAS Y HORARIOS
# ============================================================

horarios = pd.date_range(
    "2000-01-01",
    periods=int(1440 / INTERVALO_MINUTOS),
    freq=f"{INTERVALO_MINUTOS}min"
).strftime("%H:%M").tolist()

dias_texto = todos_los_dias.strftime(
    "%Y-%m-%d"
).tolist()

matriz_original = df20.pivot_table(
    index="Dia",
    columns="Hora",
    values="Caudal",
    aggfunc="mean"
)

matriz_original = matriz_original.reindex(
    index=dias_texto,
    columns=horarios
)

matriz_original = matriz_original.apply(
    pd.to_numeric,
    errors="coerce"
)

# ============================================================
# DIFERENCIAR DATOS REALES Y FALTANTES
# ============================================================

# MATRIZ ORIGINAL:
# NaN = no hay medicion.
#
# MATRIZ HEATMAP:
# NaN reemplazado por 0 para mostrarlo en blanco.
#
# ESTADISTICAS:
# Se calculan sobre MATRIZ ORIGINAL.
# Pandas excluye automaticamente los NaN.
#
# Un cero medido realmente NO es NaN,
# por lo tanto SI participa en los promedios.

mascara_faltantes = matriz_original.isna()

matriz_control = mascara_faltantes.astype(int)

matriz_heatmap = matriz_original.fillna(0)

total_intervalos = int(
    matriz_original.size
)

intervalos_reales = int(
    matriz_original.notna().sum().sum()
)

intervalos_faltantes = int(
    mascara_faltantes.sum().sum()
)

cobertura_general = (
    intervalos_reales / total_intervalos * 100
    if total_intervalos else 0
)

# ============================================================
# IDENTIFICAR PERIODO
# ============================================================

meses = {
    (fecha.year, fecha.month)
    for fecha in todos_los_dias
}

if len(meses) == 1:

    anio, mes = next(iter(meses))

    MES_EVALUADO = (
        f"{MESES_ES[mes]} {anio}"
    )

else:

    MES_EVALUADO = (
        f"{inicio:%d/%m/%Y} al {fin:%d/%m/%Y}"
    )

cartel(
    "CONTROL DE DATOS CARGADOS",
    f"""
    <b>Período:</b> {MES_EVALUADO}
    <br>
    <b>Días incluidos:</b> {len(todos_los_dias)}
    <br>
    <b>Intervalos esperados:</b> {total_intervalos}
    <br>
    <b>Intervalos con datos:</b> {intervalos_reales}
    <br>
    <b>Intervalos faltantes:</b> {intervalos_faltantes}
    <br>
    <b>Cobertura:</b> {cobertura_general:.2f}%
    <br><br>
    <b>LOS FALTANTES NO SE INCLUYEN EN LOS PROMEDIOS</b>
    """,
    "#DCFCE7"
)

# ============================================================
# PASO 2 - NOMBRE DE LOS ARCHIVOS
# ============================================================

cartel(
    "PASO 2 - NOMBRE DEL INFORME",
    """
    Escribí el nombre del informe.
    <br><br>
    Se generarán automáticamente
    un archivo <b>Excel</b> y otro <b>HTML</b>.
    <br><br>
    <b>No escribas la extensión.</b>
    """
)

nombre_base = input(
    "\n>>> NOMBRE DEL INFORME: "
).strip()

if not nombre_base:
    nombre_base = f"Analisis Caudal {MES_EVALUADO}"

nombre_base = re.sub(
    r"\.(xlsx|html)$",
    "",
    nombre_base,
    flags=re.IGNORECASE
)

nombre_base = re.sub(
    r'[<>:"/\\|?*]',
    "_",
    nombre_base
)

archivo_excel = nombre_base + ".xlsx"
archivo_html = nombre_base + ".html"

# ============================================================
# ESTADISTICAS DIARIAS
# SIN INCLUIR REGISTROS FALTANTES
# ============================================================

estadisticas = pd.DataFrame(
    index=matriz_original.index
)

# IMPORTANTE:
# Todos estos calculos utilizan la matriz ORIGINAL.
# Los NaN quedan excluidos.

estadisticas["Minimo"] = (
    matriz_original.min(axis=1)
)

estadisticas["Promedio"] = (
    matriz_original.mean(axis=1)
)

estadisticas["Maximo"] = (
    matriz_original.max(axis=1)
)

estadisticas["Rango"] = (
    estadisticas["Maximo"]
    -
    estadisticas["Minimo"]
)

estadisticas["Intervalos con datos"] = (
    matriz_original.notna().sum(axis=1)
)

estadisticas["Intervalos sin datos"] = (
    mascara_faltantes.sum(axis=1)
)

estadisticas["Cobertura (%)"] = (
    estadisticas["Intervalos con datos"]
    / len(horarios) * 100
)

estadisticas["Estado"] = np.select(
    [
        estadisticas["Intervalos con datos"] == 0,
        estadisticas["Intervalos sin datos"] == 0
    ],
    [
        "SIN DATOS",
        "COMPLETO"
    ],
    default="INCOMPLETO"
)

# ============================================================
# DURACION BAJO LIMITE
# SE CONSERVA SOLO COMO ESTADISTICA
# NO SE GENERA GRAFICO
# ============================================================

def maxima_duracion_bajo_limite(fila):

    consecutivos = 0
    maximo = 0

    for valor in fila:

        if (
            pd.notna(valor)
            and valor < LIMITE_ANOMALIA
        ):

            consecutivos += 1

            maximo = max(
                maximo,
                consecutivos
            )

        else:

            # Un faltante corta la continuidad
            consecutivos = 0

    return maximo * INTERVALO_HORAS

estadisticas["Duracion_Bajo_Limite"] = (
    matriz_original.apply(
        maxima_duracion_bajo_limite,
        axis=1
    )
)

# ============================================================
# INDICADORES GENERALES
# ============================================================

# stack() excluye NaN por defecto.
# Solo contiene mediciones reales.

valores_reales = matriz_original.stack()

minimo_general = valores_reales.min()
promedio_general = valores_reales.mean()
maximo_general = valores_reales.max()

max_evento_general = (
    estadisticas["Duracion_Bajo_Limite"].max()
)

dias_completos = int(
    (estadisticas["Estado"] == "COMPLETO").sum()
)

dias_incompletos = int(
    (estadisticas["Estado"] == "INCOMPLETO").sum()
)

dias_sin_datos = int(
    (estadisticas["Estado"] == "SIN DATOS").sum()
)

# ============================================================
# HEATMAP CORREGIDO
# CERO BLANCO
# VALOR REAL EN EL CURSOR
# UNA SOLA CAPA DE HEATMAP
# ============================================================

valores_heatmap = matriz_heatmap.to_numpy(
    dtype=float
)

positivos = valores_heatmap[
    np.isfinite(valores_heatmap)
    &
    (valores_heatmap > 0)
]

if len(positivos):

    minimo_positivo = float(positivos.min())
    maximo_positivo = float(positivos.max())

else:

    minimo_positivo = 1.0
    maximo_positivo = 1.0

zmax_color = max(maximo_positivo, 1.0)

pos_minimo = minimo_positivo / zmax_color

pos_minimo = min(
    max(pos_minimo, 0.000001),
    1.0
)

pos_blanco = pos_minimo * 0.999999

# El blanco queda reservado para cero.
# Los valores positivos empiezan en violeta.

escala_colores = [
    [0.0, "#FFFFFF"],
    [pos_blanco, "#FFFFFF"],
    [pos_minimo, "#8F8BD8"]
]

colores_restantes = [
    (0.10, "#7FA7E8"),
    (0.20, "#71C5E8"),
    (0.30, "#6EDDD0"),
    (0.40, "#91E6B0"),
    (0.50, "#C4ED8C"),
    (0.60, "#F1EE88"),
    (0.70, "#F7D487"),
    (0.80, "#F5B27F"),
    (0.90, "#F28D83"),
    (1.00, "#D97D82")
]

if pos_minimo < 1.0:

    for posicion, color in colores_restantes:

        posicion_final = (
            pos_minimo
            +
            posicion * (1.0 - pos_minimo)
        )

        escala_colores.append([
            posicion_final,
            color
        ])

# ============================================================
# INFORMACION PARA EL CURSOR
# ============================================================

estado_celdas = np.where(
    mascara_faltantes.to_numpy(),
    "FALTANTE - CERO AGREGADO",
    "MEDICION REAL"
)

fig_heatmap = go.Figure()

fig_heatmap.add_trace(
    go.Heatmap(

        z=valores_heatmap,

        x=matriz_heatmap.columns.tolist(),
        y=matriz_heatmap.index.tolist(),

        customdata=estado_celdas,

        colorscale=escala_colores,

        zmin=0,
        zmax=zmax_color,

        colorbar=dict(
            title="Caudal (m³/h)",
            thickness=18
        ),

        xgap=0,
        ygap=1,

        hovertemplate=(
            "<b>Fecha:</b> %{y}"
            "<br><b>Hora:</b> %{x}"
            "<br><b>Caudal:</b> %{z:.2f} m³/h"
            "<br><b>Origen:</b> %{customdata}"
            "<extra></extra>"
        ),

        name="Caudal"
    )
)

# ============================================================
# CRUCES NEGRAS PARA FALTANTES
# ============================================================

pos_faltantes = np.argwhere(
    mascara_faltantes.to_numpy()
)

fig_heatmap.add_trace(
    go.Scatter(

        x=[
            matriz_heatmap.columns[j]
            for i, j in pos_faltantes
        ],

        y=[
            matriz_heatmap.index[i]
            for i, j in pos_faltantes
        ],

        mode="markers",

        name="Registro faltante",

        marker=dict(
            symbol="x",
            size=6,
            color="black"
        ),

        hoverinfo="skip"
    )
)

# ============================================================
# ANOMALIAS REALES
# ============================================================

anomalias_reales = (
    ~mascara_faltantes
    &
    (matriz_original < LIMITE_ANOMALIA)
)

pos_anomalias = np.argwhere(
    anomalias_reales.to_numpy()
)

fig_heatmap.add_trace(
    go.Scatter(

        x=[
            matriz_heatmap.columns[j]
            for i, j in pos_anomalias
        ],

        y=[
            matriz_heatmap.index[i]
            for i, j in pos_anomalias
        ],

        mode="markers",

        name=f"Caudal real < {LIMITE_ANOMALIA}",

        marker=dict(
            symbol="square-open",
            size=9,
            color="red",
            line=dict(
                color="red",
                width=2
            )
        ),

        hoverinfo="skip"
    )
)

# ============================================================
# FORMATO DEL HEATMAP
# ============================================================

alto_heatmap = max(
    550,
    len(matriz_heatmap.index) * 25 + 180
)

fig_heatmap.update_layout(

    title=dict(
        text=(
            "Heatmap Operativo de Caudal"
            f"<br><sup>{MES_EVALUADO}</sup>"
        ),
        x=0.5,
        font=dict(size=22)
    ),

    xaxis_title="Hora",
    yaxis_title="Día",

    height=alto_heatmap,

    template="plotly_white",

    plot_bgcolor="#FFFFFF",
    paper_bgcolor="#FFFFFF",

    hovermode="closest",

    margin=dict(
        l=120,
        r=40,
        t=110,
        b=90
    ),

    legend=dict(
        orientation="h",
        y=1.06,
        x=1,
        xanchor="right"
    )
)

fig_heatmap.update_xaxes(
    type="category",
    tickangle=90,
    tickfont=dict(size=9),
    showgrid=False
)

fig_heatmap.update_yaxes(
    type="category",
    categoryorder="array",
    categoryarray=matriz_heatmap.index.tolist(),
    autorange="reversed",
    showgrid=False
)

# ============================================================
# GRAFICO COMPORTAMIENTO DIARIO
# PROMEDIOS SIN DATOS FALTANTES
# ============================================================

fechas_texto = estadisticas.index.tolist()

fig_estadisticas = go.Figure()

fig_estadisticas.add_trace(
    go.Scatter(
        x=fechas_texto,
        y=estadisticas["Maximo"],
        mode="lines",
        name="Máximo diario",
        line=dict(
            color="rgba(100,130,160,0.65)",
            width=1
        ),
        hoverinfo="skip"
    )
)

fig_estadisticas.add_trace(
    go.Scatter(
        x=fechas_texto,
        y=estadisticas["Minimo"],
        mode="lines",
        name="Rango mínimo-máximo",
        fill="tonexty",
        fillcolor="rgba(150,190,220,0.28)",
        line=dict(
            color="rgba(100,130,160,0.65)",
            width=1
        ),
        hoverinfo="skip"
    )
)

custom_hover = np.column_stack([
    estadisticas["Minimo"].to_numpy(),
    estadisticas["Maximo"].to_numpy(),
    estadisticas["Intervalos con datos"].to_numpy(),
    estadisticas["Intervalos sin datos"].to_numpy(),
    estadisticas["Cobertura (%)"].to_numpy()
])

fig_estadisticas.add_trace(
    go.Scatter(

        x=fechas_texto,
        y=estadisticas["Promedio"],

        mode="lines+markers",

        name="Promedio real",

        line=dict(
            color="#315F85",
            width=3
        ),

        marker=dict(size=7),

        customdata=custom_hover,

        hovertemplate=(
            "<b>Fecha:</b> %{x}"
            "<br>Mínimo: %{customdata[0]:,.2f} m³/h"
            "<br>Promedio real: %{y:,.2f} m³/h"
            "<br>Máximo: %{customdata[1]:,.2f} m³/h"
            "<br>Intervalos medidos: %{customdata[2]:.0f}"
            "<br>Intervalos faltantes: %{customdata[3]:.0f}"
            "<br>Cobertura: %{customdata[4]:.1f}%"
            "<extra></extra>"
        )
    )
)

fig_estadisticas.add_hline(
    y=LIMITE_ANOMALIA,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Límite {LIMITE_ANOMALIA}"
)

fig_estadisticas.update_layout(
    title=dict(
        text="Comportamiento Diario del Caudal",
        x=0.5
    ),
    xaxis_title="Fecha",
    yaxis_title="Caudal (m³/h)",
    height=550,
    template="plotly_white",
    hovermode="x unified",
    margin=dict(l=85, r=35, t=85, b=110)
)

fig_estadisticas.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=fechas_texto,
    tickangle=60
)

# ============================================================
# GRAFICO COBERTURA DE DATOS
# ============================================================

fig_cobertura = go.Figure()

colores_cobertura = [
    "#22C55E" if v == 100
    else "#EF4444" if v == 0
    else "#F59E0B"
    for v in estadisticas["Cobertura (%)"]
]

fig_cobertura.add_trace(
    go.Bar(

        x=fechas_texto,

        y=estadisticas["Cobertura (%)"],

        marker_color=colores_cobertura,

        name="Cobertura",

        customdata=estadisticas[
            "Intervalos sin datos"
        ],

        hovertemplate=(
            "<b>Fecha:</b> %{x}"
            "<br>Cobertura: %{y:.1f}%"
            "<br>Faltantes: %{customdata}"
            "<extra></extra>"
        )
    )
)

fig_cobertura.update_layout(
    title=dict(
        text="Cobertura Diaria de Mediciones",
        x=0.5
    ),
    xaxis_title="Fecha",
    yaxis_title="Cobertura (%)",
    yaxis=dict(range=[0, 105]),
    height=450,
    template="plotly_white",
    margin=dict(l=80, r=30, t=80, b=110)
)

fig_cobertura.update_xaxes(
    type="category",
    tickangle=60
)

# ============================================================
# TABLA RESUMEN
# ============================================================

tabla_resumen = estadisticas.copy()

tabla_resumen.index.name = "Fecha"

tabla_resumen = tabla_resumen.reset_index()

tabla_resumen = tabla_resumen.rename(
    columns={
        "Duracion_Bajo_Limite":
            "Max. horas bajo limite"
    }
)

# ============================================================
# DETALLE DE REGISTROS FALTANTES
# SOLO PARA EXCEL - NO SE INCLUYE EN HTML
# ============================================================

detalle_faltantes = matriz_control.stack()

detalle_faltantes = detalle_faltantes[
    detalle_faltantes == 1
]

detalle_faltantes = detalle_faltantes.reset_index()

detalle_faltantes.columns = [
    "Fecha",
    "Hora",
    "Faltante"
]

detalle_faltantes["Valor visualizado"] = 0

# ============================================================
# INDICADORES GENERALES
# ============================================================

control_df = pd.DataFrame(control_archivos)

indicadores_df = pd.DataFrame({
    "Indicador": [
        "Periodo",
        "Archivos procesados",
        "Registros validos",
        "Duplicados eliminados",
        "Dias del periodo",
        "Dias completos",
        "Dias incompletos",
        "Dias sin datos",
        "Intervalos esperados",
        "Intervalos medidos",
        "Intervalos faltantes",
        "Cobertura general (%)",
        "Minimo real (m3/h)",
        "Promedio real (m3/h)",
        "Maximo real (m3/h)",
        "Maxima duracion bajo limite (h)",
        "Limite de anomalia (m3/h)",
        "Intervalo de matriz (min)"
    ],
    "Valor": [
        MES_EVALUADO,
        len(control_archivos),
        len(df),
        duplicados,
        len(todos_los_dias),
        dias_completos,
        dias_incompletos,
        dias_sin_datos,
        total_intervalos,
        intervalos_reales,
        intervalos_faltantes,
        cobertura_general,
        minimo_general,
        promedio_general,
        maximo_general,
        max_evento_general,
        LIMITE_ANOMALIA,
        INTERVALO_MINUTOS
    ]
})

# ============================================================
# EXPORTAR EXCEL
# ============================================================

with pd.ExcelWriter(
    archivo_excel,
    engine="openpyxl"
) as writer:

    matriz_heatmap.to_excel(
        writer,
        sheet_name="Matriz"
    )

    matriz_original.to_excel(
        writer,
        sheet_name="Matriz original"
    )

    matriz_control.to_excel(
        writer,
        sheet_name="Faltantes 1_si"
    )

    detalle_faltantes.to_excel(
        writer,
        sheet_name="Detalle faltantes",
        index=False
    )

    tabla_resumen.to_excel(
        writer,
        sheet_name="Estadisticas",
        index=False
    )

    indicadores_df.to_excel(
        writer,
        sheet_name="Indicadores",
        index=False
    )

    control_df.to_excel(
        writer,
        sheet_name="Control archivos",
        index=False
    )

# ============================================================
# FORMATO EXCEL
# ============================================================

wb = load_workbook(archivo_excel)

for ws in wb.worksheets:

    ws.freeze_panes = (
        "B2"
        if ws.title in [
            "Matriz",
            "Matriz original",
            "Faltantes 1_si"
        ]
        else "A2"
    )

    ws.auto_filter.ref = ws.dimensions

    for celda in ws[1]:

        celda.font = Font(
            bold=True,
            color="FFFFFF"
        )

        celda.fill = PatternFill(
            "solid",
            fgColor="315F85"
        )

        celda.alignment = Alignment(
            horizontal="center"
        )

    for fila in ws.iter_rows(min_row=2):

        for celda in fila:

            if isinstance(celda.value, (int, float)):
                celda.number_format = "0.00"

    if ws.title == "Matriz":

        for i in range(matriz_heatmap.shape[0]):

            for j in range(matriz_heatmap.shape[1]):

                if mascara_faltantes.iat[i, j]:

                    celda = ws.cell(
                        row=i + 2,
                        column=j + 2
                    )

                    celda.fill = PatternFill(
                        "solid",
                        fgColor="FFF2A8"
                    )

                    celda.font = Font(
                        color="C00000",
                        bold=True
                    )

    # Anchos controlados para evitar hojas enormes
    for indice, columna in enumerate(ws.columns, start=1):

        letra = get_column_letter(indice)

        if ws.title in [
            "Matriz",
            "Matriz original",
            "Faltantes 1_si"
        ]:

            ancho = 15 if indice == 1 else 12

        else:

            largo = max(
                len(str(c.value))
                if c.value is not None else 0
                for c in columna
            )

            ancho = min(max(largo + 2, 12), 38)

        ws.column_dimensions[letra].width = ancho

    ws.row_dimensions[1].height = 28

wb.save(archivo_excel)

# ============================================================
# MOSTRAR RESULTADOS EN COLAB
# ============================================================

cartel(
    "ANÁLISIS FINALIZADO",
    f"""
    <b>Período:</b> {MES_EVALUADO}
    <br>
    <b>Días analizados:</b> {len(todos_los_dias)}
    <br>
    <b>Intervalos faltantes:</b> {intervalos_faltantes}
    <br>
    <b>Cobertura:</b> {cobertura_general:.2f}%
    <br><br>
    <b>Promedio real:</b> {promedio_general:,.2f} m³/h
    <br><br>
    Los registros faltantes NO participan
    en los promedios.
    """,
    "#DCFCE7"
)

print("\nHEATMAP DE CAUDAL")
fig_heatmap.show()

print("\nCOMPORTAMIENTO DIARIO")
fig_estadisticas.show()

print("\nCOBERTURA DE MEDICIONES")
fig_cobertura.show()

print("\nESTADÍSTICAS DIARIAS")
display(tabla_resumen.round(2))

# ============================================================
# CREAR GRAFICOS HTML
# ============================================================

html_heatmap = plot(
    fig_heatmap,
    output_type="div",
    include_plotlyjs=True
)

html_estadisticas = plot(
    fig_estadisticas,
    output_type="div",
    include_plotlyjs=False
)

html_cobertura = plot(
    fig_cobertura,
    output_type="div",
    include_plotlyjs=False
)

tabla_html = tabla_resumen.round(2).to_html(
    index=False,
    classes="tabla",
    border=0,
    na_rep="-"
)

control_html = control_df.to_html(
    index=False,
    classes="tabla",
    border=0,
    na_rep="-"
)

# ============================================================
# GENERAR INFORME HTML
# SIN TABLA DE DETALLE DE REGISTROS FALTANTES
# ============================================================

html_final = f"""
<!DOCTYPE html>
<html lang="es">

<head>

<meta charset="UTF-8">

<meta name="viewport"
content="width=device-width, initial-scale=1">

<title>{html.escape(nombre_base)}</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 25px;
    background: #f4f7fa;
    color: #263746;
}}

.contenedor {{
    max-width: 1600px;
    margin: auto;
}}

h1 {{
    text-align: center;
    color: #315F85;
    font-size: 30px;
}}

h2 {{
    color: #315F85;
    margin-top: 35px;
}}

.subtitulo {{
    text-align: center;
    font-size: 20px;
    margin-bottom: 30px;
}}

.panel {{
    background: white;
    border-radius: 14px;
    padding: 20px;
    margin-bottom: 25px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.08);
}}

.indicadores {{
    display: grid;
    grid-template-columns:
    repeat(auto-fit, minmax(200px, 1fr));
    gap: 15px;
}}

.indicador {{
    background: #e8f4fd;
    border-radius: 12px;
    padding: 20px;
    text-align: center;
}}

.valor {{
    font-size: 28px;
    font-weight: bold;
    color: #315F85;
}}

.nombre {{
    font-size: 14px;
    margin-top: 8px;
}}

.tabla {{
    border-collapse: collapse;
    width: 100%;
    font-size: 13px;
}}

.tabla th {{
    background: #315F85;
    color: white;
    padding: 10px;
    position: sticky;
    top: 0;
}}

.tabla td {{
    padding: 8px;
    text-align: center;
    border-bottom: 1px solid #ddd;
}}

.tabla tr:nth-child(even) {{
    background: #f2f6fa;
}}

.tabla-contenedor {{
    overflow-x: auto;
    max-height: 650px;
    overflow-y: auto;
}}

.nota {{
    background: #fff3cd;
    border-left: 5px solid #f59e0b;
    padding: 15px;
    margin: 20px 0;
    border-radius: 8px;
}}

.grafico {{
    width: 100%;
    overflow-x: auto;
}}

</style>
</head>

<body>

<div class="contenedor">

<h1>ANÁLISIS OPERATIVO DE CAUDAL</h1>

<div class="subtitulo">
{html.escape(MES_EVALUADO)}
</div>

<div class="panel">

<h2>Indicadores Generales</h2>

<div class="indicadores">

<div class="indicador">
<div class="valor">{len(todos_los_dias)}</div>
<div class="nombre">Días del período</div>
</div>

<div class="indicador">
<div class="valor">{dias_completos}</div>
<div class="nombre">Días completos</div>
</div>

<div class="indicador">
<div class="valor">{dias_incompletos}</div>
<div class="nombre">Días incompletos</div>
</div>

<div class="indicador">
<div class="valor">{dias_sin_datos}</div>
<div class="nombre">Días sin datos</div>
</div>

<div class="indicador">
<div class="valor">{intervalos_faltantes}</div>
<div class="nombre">Intervalos faltantes</div>
</div>

<div class="indicador">
<div class="valor">{cobertura_general:.1f}%</div>
<div class="nombre">Cobertura general</div>
</div>

<div class="indicador">
<div class="valor">{minimo_general:,.0f}</div>
<div class="nombre">Mínimo real m³/h</div>
</div>

<div class="indicador">
<div class="valor">{promedio_general:,.0f}</div>
<div class="nombre">Promedio real m³/h</div>
</div>

<div class="indicador">
<div class="valor">{maximo_general:,.0f}</div>
<div class="nombre">Máximo real m³/h</div>
</div>

</div>
</div>

<div class="nota">

<b>CRITERIO DE CÁLCULO</b>

<br><br>

Los intervalos sin mediciones se representan
con cero en el heatmap, pero NO se incluyen
en los cálculos de promedio, mínimo y máximo.

<br><br>

Los ceros realmente medidos sí se incluyen.

<br><br>

<b>Blanco:</b> caudal cero.

<br>

<b>Cruz negra:</b> registro faltante.

<br>

<b>Cuadrado rojo:</b> medición real bajo límite.

<br><br>

Los promedios diarios utilizan únicamente
los intervalos con mediciones reales.

</div>

<div class="panel">
<h2>1. Heatmap Operativo de Caudal</h2>
<div class="grafico">
{html_heatmap}
</div>
</div>

<div class="panel">
<h2>2. Comportamiento Diario del Caudal</h2>
<div class="grafico">
{html_estadisticas}
</div>
</div>

<div class="panel">
<h2>3. Cobertura Diaria de Mediciones</h2>
<div class="grafico">
{html_cobertura}
</div>
</div>

<div class="panel">
<h2>4. Tabla Resumen Diaria</h2>
<div class="tabla-contenedor">
{tabla_html}
</div>
</div>

<div class="panel">
<h2>5. Control de Archivos</h2>
<div class="tabla-contenedor">
{control_html}
</div>
</div>

</div>
</body>
</html>
"""

# ============================================================
# GUARDAR HTML
# ============================================================

with open(
    archivo_html,
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(html_final)

# ============================================================
# CONTROL FINAL
# ============================================================

print("\n" + "=" * 65)
print("CONTROL FINAL DEL ANALISIS")
print("=" * 65)

print("Periodo:", MES_EVALUADO)
print("Dias:", len(todos_los_dias))
print("Intervalos reales:", intervalos_reales)
print("Intervalos faltantes:", intervalos_faltantes)
print(f"Cobertura: {cobertura_general:.2f}%")

print()
print("Minimo real:", minimo_general)
print("Promedio real:", promedio_general)
print("Maximo real:", maximo_general)

print()
print("Promedios calculados sin registros faltantes.")
print("Grafico de duracion bajo limite eliminado.")
print("Detalle de faltantes eliminado del HTML.")
print("Detalle de faltantes conservado en Excel.")

print("=" * 65)

# ============================================================
# DESCARGAR ARCHIVOS
# ============================================================

cartel(
    "ARCHIVOS GENERADOS",
    f"""
    <b>{html.escape(archivo_excel)}</b>
    <br><br>
    <b>{html.escape(archivo_html)}</b>
    <br><br>
    Se iniciará la descarga automática.
    """,
    "#DCFCE7"
)

files.download(archivo_excel)
files.download(archivo_html)

print("\nPROCESO COMPLETADO CORRECTAMENTE")


# ***COMPILADRO DE TOMAS DE PHINDOWS DESDE VISUALIZADOR***

In [24]:
# @title

!pip install -q plotly openpyxl

import io
import re
import html
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from google.colab import files
from IPython.display import display, HTML
from plotly.offline import plot

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# 1. CONFIGURACION
# ============================================================

LIMITE_ANOMALIA = 4200
FORMATO_FECHA = "DMY"       # DMY o MDY
TOLERANCIA_RELATIVA = 0.10
MAX_POSICIONES = 1000000

def cartel(titulo, mensaje, color="#FFF3CD"):
    display(HTML(f"""
    <div style="background:{color};border:3px solid #D6A000;
    border-radius:14px;padding:22px;margin:18px 0;
    text-align:center;font-family:Arial">
    <div style="font-size:27px;font-weight:bold">
    {html.escape(str(titulo))}</div>
    <div style="font-size:20px;line-height:1.6;margin-top:12px">
    {mensaje}</div></div>
    """))

def intervalo_texto(segundos):
    if pd.isna(segundos):
        return "No determinado"
    s = int(round(float(segundos)))
    h, resto = divmod(s, 3600)
    m, seg = divmod(resto, 60)
    if h:
        return f"{h} h {m} min {seg} s"
    return f"{m} min {seg} s"

# ============================================================
# 2. CONVERTIR DATOS
# ============================================================

def convertir_numeros(serie):
    s = serie.astype("string").str.strip()
    s = s.str.replace(" ", "", regex=False)

    europeo = s.str.match(
        r"^-?\d{1,3}(\.\d{3})+,\d+$", na=False
    )
    s.loc[europeo] = (
        s.loc[europeo]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    coma = (
        s.str.contains(",", regex=False, na=False)
        & ~s.str.contains(".", regex=False, na=False)
    )
    s.loc[coma] = s.loc[coma].str.replace(
        ",", ".", regex=False
    )

    return pd.to_numeric(s, errors="coerce")

def convertir_fechas(serie):
    s = serie.astype("string").str.strip()

    resultado = pd.Series(
        pd.NaT, index=s.index, dtype="datetime64[ns]"
    )

    if FORMATO_FECHA == "DMY":
        formatos = [
            "%d/%m/%Y %H:%M:%S",
            "%d/%m/%Y %H:%M",
            "%d/%m/%Y",
            "%d-%m-%Y %H:%M:%S",
            "%d-%m-%Y",
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d %H:%M",
            "%Y-%m-%d"
        ]
    else:
        formatos = [
            "%m/%d/%Y %H:%M:%S",
            "%m/%d/%Y %H:%M",
            "%m/%d/%Y",
            "%m-%d-%Y %H:%M:%S",
            "%m-%d-%Y",
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d %H:%M",
            "%Y-%m-%d"
        ]

    for formato in formatos:
        pendiente = resultado.isna() & s.notna()
        if not pendiente.any():
            break
        resultado.loc[pendiente] = pd.to_datetime(
            s.loc[pendiente],
            format=formato,
            errors="coerce"
        )

    pendiente = resultado.isna() & s.notna()
    if pendiente.any():
        resultado.loc[pendiente] = pd.to_datetime(
            s.loc[pendiente],
            format="mixed",
            dayfirst=(FORMATO_FECHA == "DMY"),
            errors="coerce"
        )

    return resultado

def convertir_horas(serie):
    s = serie.astype("string").str.strip()
    s = s.str.replace(
        r"^(\d{1,2}:\d{2})$",
        r"\1:00",
        regex=True
    )

    resultado = pd.to_timedelta(s, errors="coerce")

    pendiente = resultado.isna()
    if pendiente.any():
        n = convertir_numeros(s.loc[pendiente])
        validos = n.between(0, 1, inclusive="left")
        indices = n.index[validos]
        resultado.loc[indices] = pd.to_timedelta(
            n.loc[indices], unit="D"
        )

    return resultado

# ============================================================
# 3. LEER CSV Y EXCEL
# ============================================================

def leer_archivo(nombre, contenido):
    if nombre.lower().endswith(".csv"):
        ultimo_error = None
        for codificacion in ["utf-8-sig", "latin-1"]:
            try:
                return pd.read_csv(
                    io.BytesIO(contenido),
                    header=None,
                    dtype=str,
                    sep=None,
                    engine="python",
                    encoding=codificacion,
                    skip_blank_lines=True
                )
            except Exception as e:
                ultimo_error = e
        raise ValueError(str(ultimo_error))

    if nombre.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(
            io.BytesIO(contenido),
            header=None,
            dtype=str
        )

    raise ValueError("Solo CSV y Excel")

def normalizar_archivo(nombre, contenido):
    bruto = leer_archivo(nombre, contenido)
    bruto = bruto.dropna(how="all").reset_index(drop=True)

    if bruto.shape[1] < 2:
        raise ValueError("Se necesitan al menos 2 columnas")

    encabezado = str(bruto.iloc[0, 0]).strip().lower()

    es_tiempo = encabezado in [
        "tiempo", "timestamp", "datetime",
        "fecha hora", "fecha_hora"
    ]
    es_fecha = encabezado in ["fecha", "date"]

    if es_tiempo or bruto.shape[1] == 2:
        formato = "Tiempo + Caudal"
        inicio = 1 if es_tiempo else 0
        datos = bruto.iloc[inicio:, :2].copy()
        datos.columns = ["Tiempo", "Caudal"]
        datos["FechaHora"] = convertir_fechas(
            datos["Tiempo"]
        )
    else:
        formato = "Fecha + Hora + Caudal"
        inicio = 1 if es_fecha else 0
        datos = bruto.iloc[inicio:, :3].copy()
        datos.columns = ["Fecha", "Hora", "Caudal"]

        fechas = convertir_fechas(datos["Fecha"])
        horas = convertir_horas(datos["Hora"])
        base = fechas.dt.normalize()
        datos["FechaHora"] = base + horas

        con_hora = fechas.notna() & (fechas != base)
        datos.loc[con_hora, "FechaHora"] = fechas.loc[
            con_hora
        ]

    originales = len(datos)
    datos["Caudal"] = convertir_numeros(datos["Caudal"])

    fechas_invalidas = int(datos["FechaHora"].isna().sum())
    caudales_invalidos = int(datos["Caudal"].isna().sum())

    datos = datos.dropna(
        subset=["FechaHora", "Caudal"]
    ).copy()

    datos["Fecha"] = datos["FechaHora"].dt.strftime(
        "%d/%m/%Y"
    )
    datos["Hora"] = datos["FechaHora"].dt.strftime(
        "%H:%M:%S"
    )
    datos["Archivo"] = nombre

    control = {
        "Archivo": nombre,
        "Formato": formato,
        "Registros originales": originales,
        "Registros validos": len(datos),
        "Descartados": originales - len(datos),
        "Fechas invalidas": fechas_invalidas,
        "Caudales invalidos": caudales_invalidos
    }
    return datos, control

# ============================================================
# 4. CARGAR ARCHIVOS
# ============================================================

cartel(
    "PASO 1 - CARGAR ARCHIVOS",
    """
    Seleccioná los CSV o Excel de caudal.
    <br><br>
    <b>El intervalo de muestreo será detectado
    automáticamente y utilizado para reconstruir
    el heatmap.</b>
    """
)

uploaded = files.upload()
if not uploaded:
    raise ValueError("No se cargaron archivos")

grupos = []
controles = []

for nombre, contenido in uploaded.items():
    try:
        datos, control = normalizar_archivo(
            nombre, contenido
        )
        controles.append(control)
        if len(datos):
            grupos.append(datos)
        print(f"{nombre}: {len(datos)} registros válidos")
    except Exception as e:
        print(f"ERROR EN {nombre}: {e}")

if not grupos:
    raise ValueError("No hay datos válidos")

df = pd.concat(grupos, ignore_index=True)
df = df.sort_values("FechaHora").reset_index(drop=True)

duplicados = int(
    df.duplicated(["FechaHora", "Caudal"]).sum()
)
df = df.drop_duplicates(
    ["FechaHora", "Caudal"]
).reset_index(drop=True)

serie = (
    df.groupby("FechaHora")["Caudal"]
    .mean()
    .sort_index()
)

if len(serie) < 3:
    raise ValueError(
        "Se necesitan al menos tres mediciones"
    )

# ============================================================
# 5. DETECTAR INTERVALO
# ============================================================

diferencias = (
    serie.index.to_series()
    .diff()
    .dt.total_seconds()
    .dropna()
)
diferencias = diferencias[diferencias > 0]

if diferencias.empty:
    raise ValueError("No se pudo detectar el intervalo")

# Agrupacion en ventanas de 5 segundos.
grupos_intervalo = (diferencias / 5).round() * 5
moda = grupos_intervalo.mode().iloc[0]

candidatos = diferencias[
    grupos_intervalo == moda
]

intervalo_detectado = float(candidatos.median())
intervalo_reconstruccion = float(candidatos.mean())

if intervalo_reconstruccion <= 0:
    raise ValueError("Intervalo inválido")

porcentaje_predominante = (
    len(candidatos) / len(diferencias) * 100
)

tolerancia = max(
    2.0,
    intervalo_reconstruccion * TOLERANCIA_RELATIVA
)

control_intervalos = []

for nombre_archivo in df["Archivo"].unique():
    tiempos = (
        df.loc[
            df["Archivo"] == nombre_archivo,
            "FechaHora"
        ]
        .drop_duplicates()
        .sort_values()
    )

    dif = tiempos.diff().dt.total_seconds().dropna()
    dif = dif[dif > 0]

    if len(dif):
        grupos_dif = (dif / 5).round() * 5
        moda_dif = grupos_dif.mode().iloc[0]
        estimacion = float(
            dif[grupos_dif == moda_dif].mean()
        )
    else:
        estimacion = np.nan

    control_intervalos.append({
        "Archivo": nombre_archivo,
        "Intervalo (s)": estimacion,
        "Intervalo detectado": intervalo_texto(
            estimacion
        )
    })

control_intervalos = pd.DataFrame(
    control_intervalos
)

cartel(
    "INTERVALO DETECTADO",
    f"""
    <b style="font-size:30px">
    {intervalo_texto(intervalo_detectado)}
    </b>
    <br>
    Intervalo medio: {intervalo_reconstruccion:.2f} s
    <br>
    Coincidencia: {porcentaje_predominante:.1f}%
    """,
    "#DCFCE7"
)

display(control_intervalos)

if porcentaje_predominante < 50:
    print(
        "ADVERTENCIA: existen diferencias importantes "
        "entre los intervalos. Revisar el muestreo."
    )

# ============================================================
# 6. RECONSTRUIR SECUENCIA REAL
# ============================================================

origen = serie.index.min()

segundos = (
    serie.index - origen
).total_seconds()

posiciones = np.rint(
    segundos / intervalo_reconstruccion
).astype(int)

desviaciones = np.abs(
    segundos - posiciones * intervalo_reconstruccion
)

alineadas = desviaciones <= tolerancia

lecturas = pd.DataFrame({
    "FechaHora": serie.index,
    "Caudal": serie.to_numpy(),
    "Posicion": posiciones,
    "Desviacion_s": desviaciones,
    "Alineada": alineadas
})

lecturas_validas = lecturas.loc[
    lecturas["Alineada"]
].copy()

lecturas_no_alineadas = lecturas.loc[
    ~lecturas["Alineada"]
].copy()

if lecturas_validas.empty:
    raise ValueError(
        "No hay mediciones alineadas con "
        "el intervalo detectado"
    )

ultima_posicion = int(posiciones.max())

if ultima_posicion > MAX_POSICIONES:
    raise ValueError(
        "Demasiadas posiciones esperadas. "
        "Revisá las fechas y el intervalo."
    )

indice = pd.RangeIndex(
    0, ultima_posicion + 1,
    name="Posicion"
)

secuencia = pd.DataFrame(index=indice)

secuencia["Tiempo esperado"] = (
    origen
    + pd.to_timedelta(
        secuencia.index.to_numpy()
        * intervalo_reconstruccion,
        unit="s"
    )
)

agrupado = lecturas_validas.groupby(
    "Posicion"
).agg(
    Caudal=("Caudal", "mean"),
    Tiempo_real=("FechaHora", "first"),
    Cantidad=("Caudal", "size")
)

secuencia = secuencia.join(agrupado)

secuencia["Faltante"] = secuencia["Caudal"].isna()

secuencia["Fecha"] = (
    secuencia["Tiempo esperado"]
    .dt.strftime("%Y-%m-%d")
)

secuencia["Hora esperada"] = (
    secuencia["Tiempo esperado"]
    .dt.strftime("%H:%M:%S")
)

# ============================================================
# 7. MATRIZ CON UNA COLUMNA POR MUESTRA ESPERADA
# ============================================================
#
# CORRECCION PRINCIPAL:
#
# ANTES:
#   columnas fijas de 20 minutos
#
# AHORA:
#   cada columna corresponde a una muestra
#   de la frecuencia REAL detectada.
#
# La secuencia se reconstruye continuamente,
# y despues se distribuye por dia.
#
# No se agregan posiciones artificiales de 20 min.
#
# ============================================================

secuencia["Muestra del dia"] = (
    secuencia.groupby("Fecha").cumcount() + 1
)

inicio = serie.index.min().normalize()
fin = serie.index.max().normalize()

todos_los_dias = pd.date_range(
    inicio, fin, freq="D"
)

dias_texto = todos_los_dias.strftime(
    "%Y-%m-%d"
).tolist()

max_muestras_dia = int(
    secuencia["Muestra del dia"].max()
)

columnas = list(
    range(1, max_muestras_dia + 1)
)

matriz = secuencia.pivot(
    index="Fecha",
    columns="Muestra del dia",
    values="Caudal"
).reindex(
    index=dias_texto,
    columns=columnas
)

matriz_tiempos = secuencia.pivot(
    index="Fecha",
    columns="Muestra del dia",
    values="Tiempo esperado"
).reindex(
    index=dias_texto,
    columns=columnas
)

matriz_tiempos_reales = secuencia.pivot(
    index="Fecha",
    columns="Muestra del dia",
    values="Tiempo_real"
).reindex(
    index=dias_texto,
    columns=columnas
)

# Celdas donde realmente se esperaba una medicion.
esperadas = matriz_tiempos.notna()

# Solo son faltantes las posiciones esperadas
# sin caudal registrado.
faltantes = esperadas & matriz.isna()

matriz_control = faltantes.astype(int)

total_esperados = int(esperadas.sum().sum())
total_medidos = int(matriz.notna().sum().sum())
total_faltantes = int(faltantes.sum().sum())

cobertura = (
    total_medidos / total_esperados * 100
    if total_esperados else 0
)

# ============================================================
# 8. ETIQUETAS HORARIAS PARA EL EJE X
# ============================================================
#
# La hora esperada puede variar entre dias.
# Se utiliza la hora mediana de cada posicion
# para etiquetar el eje X.
#
# En el cursor se muestra la hora exacta
# de cada dia y cada muestra.
#
# ============================================================

etiquetas_horas = []

for columna in columnas:
    tiempos = matriz_tiempos[columna].dropna()

    if len(tiempos):
        segundos_dia = (
            tiempos.dt.hour * 3600
            + tiempos.dt.minute * 60
            + tiempos.dt.second
        )

        segundos_referencia = int(
            np.median(segundos_dia)
        )

        hh = segundos_referencia // 3600
        mm = (segundos_referencia % 3600) // 60

        etiqueta = f"{hh:02d}:{mm:02d}"
    else:
        etiqueta = ""

    etiquetas_horas.append(etiqueta)

# Para evitar etiquetas repetidas o confusas,
# el eje X usa posiciones numericas y muestra
# las horas representativas como etiquetas.

# ============================================================
# 9. PERIODO
# ============================================================

if (
    inicio.year == fin.year
    and inicio.month == fin.month
):
    periodo = inicio.strftime("%m/%Y")
else:
    periodo = (
        f"{inicio:%d/%m/%Y}"
        f" al {fin:%d/%m/%Y}"
    )

# ============================================================
# 10. ESTADISTICAS SOBRE DATOS REALES
# ============================================================

datos_estadisticas = serie.reset_index()
datos_estadisticas.columns = [
    "FechaHora", "Caudal"
]

datos_estadisticas["Fecha"] = (
    datos_estadisticas["FechaHora"]
    .dt.strftime("%Y-%m-%d")
)

estadisticas = (
    datos_estadisticas.groupby("Fecha")["Caudal"]
    .agg(
        Minimo="min",
        Promedio="mean",
        Maximo="max",
        Mediciones="count"
    )
    .reindex(dias_texto)
)

estadisticas.index.name = "Fecha"

estadisticas["Rango"] = (
    estadisticas["Maximo"]
    - estadisticas["Minimo"]
)

estadisticas["Esperados"] = esperadas.sum(axis=1)
estadisticas["Medidos alineados"] = matriz.notna().sum(
    axis=1
)
estadisticas["Faltantes"] = faltantes.sum(axis=1)

estadisticas["Cobertura (%)"] = np.where(
    estadisticas["Esperados"] > 0,
    estadisticas["Medidos alineados"]
    / estadisticas["Esperados"] * 100,
    0
)

estadisticas["Estado"] = np.select(
    [
        estadisticas["Mediciones"].fillna(0) == 0,
        estadisticas["Faltantes"] == 0
    ],
    [
        "SIN DATOS",
        "COMPLETO"
    ],
    default="INCOMPLETO"
)

def duracion_bajo_limite(fila):
    maximo = 0
    actual = 0

    for valor in fila:
        if pd.notna(valor) and valor < LIMITE_ANOMALIA:
            actual += 1
            maximo = max(maximo, actual)
        else:
            actual = 0

    return maximo * intervalo_reconstruccion / 3600

estadisticas["Max horas bajo limite"] = matriz.apply(
    duracion_bajo_limite,
    axis=1
)

minimo_general = serie.min()
promedio_general = serie.mean()
maximo_general = serie.max()

dias_completos = int(
    (estadisticas["Estado"] == "COMPLETO").sum()
)
dias_incompletos = int(
    (estadisticas["Estado"] == "INCOMPLETO").sum()
)
dias_sin_datos = int(
    (estadisticas["Estado"] == "SIN DATOS").sum()
)

cartel(
    "RECONSTRUCCIÓN COMPLETADA",
    f"""
    <b>Intervalo:</b>
    {intervalo_texto(intervalo_detectado)}
    <br>
    <b>Días:</b> {len(dias_texto)}
    <br>
    <b>Máximo de muestras por día:</b>
    {max_muestras_dia}
    <br>
    <b>Posiciones esperadas:</b>
    {total_esperados}
    <br>
    <b>Posiciones medidas:</b>
    {total_medidos}
    <br>
    <b>Faltantes:</b>
    {total_faltantes}
    <br>
    <b>Cobertura:</b>
    {cobertura:.2f}%
    """,
    "#DCFCE7"
)

# ============================================================
# 11. NOMBRE DEL INFORME
# ============================================================

cartel(
    "PASO 2 - NOMBRE DEL INFORME",
    """
    Escribí el nombre del informe.
    <br><br>
    Se generarán Excel y HTML.
    <br><br>
    <b>No incluyas la extensión.</b>
    """
)

nombre = input(
    "\n>>> NOMBRE DEL INFORME: "
).strip()

if not nombre:
    nombre = "Analisis_Caudal"

nombre = re.sub(
    r"\.(xlsx|html)$",
    "",
    nombre,
    flags=re.IGNORECASE
)

nombre = re.sub(
    r'[<>:"/\\|?*]',
    "_",
    nombre
)

archivo_excel = nombre + ".xlsx"
archivo_html = nombre + ".html"

# ============================================================
# 12. HEATMAP CON INTERVALO REAL
# ============================================================

z = matriz.to_numpy(dtype=float)

horas_cursor = np.empty(
    matriz.shape, dtype=object
)

for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):

        real = matriz_tiempos_reales.iat[i, j]
        esperado = matriz_tiempos.iat[i, j]

        if pd.notna(real):
            horas_cursor[i, j] = (
                "Hora real: "
                + pd.Timestamp(real).strftime(
                    "%H:%M:%S"
                )
            )
        elif pd.notna(esperado):
            horas_cursor[i, j] = (
                "Faltante. Esperado: "
                + pd.Timestamp(esperado).strftime(
                    "%H:%M:%S"
                )
            )
        else:
            horas_cursor[i, j] = (
                "Sin posición esperada"
            )

fig_heatmap = go.Figure()

fig_heatmap.add_trace(
    go.Heatmap(
        z=z,
        x=columnas,
        y=dias_texto,
        customdata=horas_cursor,
        colorscale="Turbo",
        colorbar=dict(
            title="Caudal (m³/h)",
            thickness=18
        ),
        xgap=1,
        ygap=1,
        hovertemplate=(
            "<b>Fecha:</b> %{y}"
            "<br><b>Muestra del día:</b> %{x}"
            "<br><b>Caudal:</b> %{z:.2f} m³/h"
            "<br>%{customdata}"
            "<extra></extra>"
        ),
        name="Caudal"
    )
)

# Cruces negras para faltantes reales.
pos_faltantes = np.argwhere(
    faltantes.to_numpy()
)

fig_heatmap.add_trace(
    go.Scatter(
        x=[
            columnas[j]
            for i, j in pos_faltantes
        ],
        y=[
            dias_texto[i]
            for i, j in pos_faltantes
        ],
        mode="markers",
        name="Registro faltante",
        marker=dict(
            symbol="x",
            size=8,
            color="black",
            line=dict(width=1.5)
        ),
        hoverinfo="skip"
    )
)

# Cuadrados rojos para valores reales bajo limite.
anomalias = matriz.notna() & (
    matriz < LIMITE_ANOMALIA
)

pos_anomalias = np.argwhere(
    anomalias.to_numpy()
)

fig_heatmap.add_trace(
    go.Scatter(
        x=[
            columnas[j]
            for i, j in pos_anomalias
        ],
        y=[
            dias_texto[i]
            for i, j in pos_anomalias
        ],
        mode="markers",
        name=f"Caudal < {LIMITE_ANOMALIA}",
        marker=dict(
            symbol="square-open",
            size=9,
            color="red",
            line=dict(width=2, color="red")
        ),
        hoverinfo="skip"
    )
)

fig_heatmap.update_layout(
    title=dict(
        text=(
            "Heatmap Operativo de Caudal"
            f"<br><sup>{periodo} | "
            f"Muestreo: "
            f"{intervalo_texto(intervalo_detectado)}"
            "</sup>"
        ),
        x=0.5,
        font=dict(size=22)
    ),
    xaxis_title="Hora",
    yaxis_title="Día",
    height=max(
        550,
        len(dias_texto) * 25 + 180
    ),
    template="plotly_white",
    plot_bgcolor="#FFFFFF",
    paper_bgcolor="#FFFFFF",
    hovermode="closest",
    margin=dict(
        l=120, r=40, t=110, b=100
    ),
    legend=dict(
        orientation="h",
        y=1.06,
        x=1,
        xanchor="right"
    )
)

# El eje X muestra las horas de las muestras
# y NO una escala fija de 20 minutos.

fig_heatmap.update_xaxes(
    type="linear",
    tickmode="array",
    tickvals=columnas,
    ticktext=etiquetas_horas,
    tickangle=90,
    tickfont=dict(size=10),
    showgrid=False,
    range=[
        columnas[0] - 0.5,
        columnas[-1] + 0.5
    ]
)

fig_heatmap.update_yaxes(
    type="category",
    categoryorder="array",
    categoryarray=dias_texto,
    autorange="reversed",
    showgrid=False
)

# ============================================================
# 13. COMPORTAMIENTO DIARIO - DISEÑO ORIGINAL
# ============================================================

fig_diario = go.Figure()

# Maximo: borde superior de la banda.
fig_diario.add_trace(
    go.Scatter(
        x=dias_texto,
        y=estadisticas["Maximo"],
        mode="lines",
        name="Máximo diario",
        line=dict(
            color="rgba(95,137,170,0.70)",
            width=1
        ),
        hoverinfo="skip",
        showlegend=False
    )
)

# Minimo: borde inferior y relleno celeste.
fig_diario.add_trace(
    go.Scatter(
        x=dias_texto,
        y=estadisticas["Minimo"],
        mode="lines",
        name="Rango mínimo-máximo",
        fill="tonexty",
        fillcolor="rgba(150,190,220,0.28)",
        line=dict(
            color="rgba(95,137,170,0.70)",
            width=1
        ),
        hoverinfo="skip"
    )
)

datos_cursor = np.column_stack([
    estadisticas["Minimo"].to_numpy(),
    estadisticas["Maximo"].to_numpy(),
    estadisticas["Mediciones"].fillna(0).to_numpy(),
    estadisticas["Cobertura (%)"].to_numpy()
])

# Promedio diario: linea azul gruesa.
fig_diario.add_trace(
    go.Scatter(
        x=dias_texto,
        y=estadisticas["Promedio"],
        mode="lines+markers",
        name="Promedio diario",
        line=dict(
            color="#315F85",
            width=3.5
        ),
        marker=dict(
            color="#315F85",
            size=8,
            symbol="circle"
        ),
        customdata=datos_cursor,
        hovertemplate=(
            "<b>Fecha:</b> %{x}"
            "<br>Mínimo: %{customdata[0]:,.2f} m³/h"
            "<br>Promedio: %{y:,.2f} m³/h"
            "<br>Máximo: %{customdata[1]:,.2f} m³/h"
            "<br>Mediciones: %{customdata[2]:.0f}"
            "<br>Cobertura: %{customdata[3]:.1f}%"
            "<extra></extra>"
        )
    )
)

fig_diario.add_hline(
    y=LIMITE_ANOMALIA,
    line_color="red",
    line_width=2,
    line_dash="dash",
    annotation_text=(
        f"Límite {LIMITE_ANOMALIA} m³/h"
    ),
    annotation_position="bottom left"
)

fig_diario.update_layout(
    title=dict(
        text="Comportamiento Diario del Caudal",
        x=0.5,
        font=dict(
            size=21,
            color="#173B67"
        )
    ),
    xaxis_title="Fecha",
    yaxis_title="Caudal (m³/h)",
    height=600,
    template="plotly_white",
    plot_bgcolor="#FFFFFF",
    paper_bgcolor="#FFFFFF",
    hovermode="x unified",
    margin=dict(
        l=90, r=50, t=95, b=115
    ),
    legend=dict(
        orientation="v",
        x=1.02,
        y=1
    )
)

fig_diario.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=dias_texto,
    tickangle=60,
    showgrid=True,
    gridcolor="#E5EDF5"
)

fig_diario.update_yaxes(
    showgrid=True,
    gridcolor="#E5EDF5",
    zeroline=False
)

# ============================================================
# 14. COBERTURA DIARIA
# ============================================================

colores_cobertura = [
    "#22C55E" if v >= 99.999
    else "#EF4444" if v == 0
    else "#F59E0B"
    for v in estadisticas["Cobertura (%)"]
]

fig_cobertura = go.Figure()

fig_cobertura.add_trace(
    go.Bar(
        x=dias_texto,
        y=estadisticas["Cobertura (%)"],
        marker_color=colores_cobertura,
        name="Cobertura",
        customdata=estadisticas["Faltantes"],
        hovertemplate=(
            "<b>Fecha:</b> %{x}"
            "<br>Cobertura: %{y:.1f}%"
            "<br>Faltantes: %{customdata}"
            "<extra></extra>"
        )
    )
)

fig_cobertura.update_layout(
    title=dict(
        text="Cobertura Diaria de Mediciones",
        x=0.5
    ),
    xaxis_title="Fecha",
    yaxis_title="Cobertura (%)",
    yaxis=dict(range=[0, 105]),
    height=450,
    template="plotly_white",
    margin=dict(
        l=80, r=30, t=80, b=110
    )
)

fig_cobertura.update_xaxes(
    type="category",
    tickangle=60
)

# ============================================================
# 15. PREPARAR TABLAS
# ============================================================

datos_normalizados = df[
    ["Fecha", "Hora", "Caudal", "Archivo"]
].copy()

# CORRECCION DEL ERROR DE POSICION DUPLICADA:
# Se utiliza reset_index() una sola vez.
# No se renombra otra columna como Posicion.

detalle_faltantes = (
    secuencia.loc[
        secuencia["Faltante"],
        [
            "Tiempo esperado",
            "Fecha",
            "Hora esperada",
            "Muestra del dia"
        ]
    ]
    .reset_index()
)

resumen = estadisticas.reset_index()

indicadores = pd.DataFrame({
    "Indicador": [
        "Periodo",
        "Intervalo detectado (s)",
        "Intervalo detectado",
        "Intervalo medio (s)",
        "Coincidencia predominante (%)",
        "Archivos",
        "Registros normalizados",
        "Instantes distintos",
        "Duplicados eliminados",
        "Mediciones no alineadas",
        "Dias",
        "Dias completos",
        "Dias incompletos",
        "Dias sin datos",
        "Posiciones esperadas",
        "Posiciones medidas",
        "Posiciones faltantes",
        "Cobertura (%)",
        "Minimo real (m3/h)",
        "Promedio real (m3/h)",
        "Maximo real (m3/h)",
        "Limite (m3/h)"
    ],
    "Valor": [
        periodo,
        intervalo_detectado,
        intervalo_texto(intervalo_detectado),
        intervalo_reconstruccion,
        porcentaje_predominante,
        len(grupos),
        len(df),
        len(serie),
        duplicados,
        len(lecturas_no_alineadas),
        len(dias_texto),
        dias_completos,
        dias_incompletos,
        dias_sin_datos,
        total_esperados,
        total_medidos,
        total_faltantes,
        cobertura,
        minimo_general,
        promedio_general,
        maximo_general,
        LIMITE_ANOMALIA
    ]
})

control_archivos = pd.DataFrame(controles)

# Columnas para Excel con la hora de referencia.
columnas_excel = [
    f"{i:02d} - {hora}"
    for i, hora in zip(
        columnas, etiquetas_horas
    )
]

matriz_excel = matriz.copy()
matriz_excel.columns = columnas_excel

horas_excel = matriz_tiempos_reales.copy()
horas_excel.columns = columnas_excel

esperadas_excel = matriz_tiempos.copy()
esperadas_excel.columns = columnas_excel

control_excel = matriz_control.copy()
control_excel.columns = columnas_excel

# ============================================================
# 16. EXPORTAR EXCEL
# ============================================================

with pd.ExcelWriter(
    archivo_excel,
    engine="openpyxl"
) as writer:

    datos_normalizados.to_excel(
        writer,
        sheet_name="Datos normalizados",
        index=False
    )

    matriz_excel.to_excel(
        writer,
        sheet_name="Matriz"
    )

    horas_excel.to_excel(
        writer,
        sheet_name="Horas reales"
    )

    esperadas_excel.to_excel(
        writer,
        sheet_name="Horas esperadas"
    )

    control_excel.to_excel(
        writer,
        sheet_name="Faltantes 1_si"
    )

    detalle_faltantes.to_excel(
        writer,
        sheet_name="Detalle faltantes",
        index=False
    )

    resumen.to_excel(
        writer,
        sheet_name="Estadisticas",
        index=False
    )

    indicadores.to_excel(
        writer,
        sheet_name="Indicadores",
        index=False
    )

    control_archivos.to_excel(
        writer,
        sheet_name="Control archivos",
        index=False
    )

    control_intervalos.to_excel(
        writer,
        sheet_name="Muestreo por archivo",
        index=False
    )

    lecturas_no_alineadas.to_excel(
        writer,
        sheet_name="Mediciones no alineadas",
        index=False
    )

    secuencia.reset_index().to_excel(
        writer,
        sheet_name="Secuencia esperada",
        index=False
    )

# ============================================================
# 17. FORMATO EXCEL
# ============================================================

wb = load_workbook(archivo_excel)

for ws in wb.worksheets:

    ws.freeze_panes = (
        "B2"
        if ws.title in [
            "Matriz",
            "Horas reales",
            "Horas esperadas",
            "Faltantes 1_si"
        ]
        else "A2"
    )

    ws.auto_filter.ref = ws.dimensions

    for celda in ws[1]:
        celda.font = Font(
            bold=True,
            color="FFFFFF"
        )
        celda.fill = PatternFill(
            "solid",
            fgColor="315F85"
        )
        celda.alignment = Alignment(
            horizontal="center"
        )

    for indice in range(
        1, ws.max_column + 1
    ):
        letra = get_column_letter(indice)

        ws.column_dimensions[letra].width = (
            18 if indice == 1 else 20
        )

    if ws.title in [
        "Horas reales",
        "Horas esperadas"
    ]:
        for fila in ws.iter_rows(
            min_row=2,
            min_col=2
        ):
            for celda in fila:
                if celda.value is not None:
                    celda.number_format = (
                        "dd/mm/yyyy hh:mm:ss"
                    )

    if ws.title == "Matriz":
        for i in range(matriz.shape[0]):
            for j in range(matriz.shape[1]):
                if faltantes.iat[i, j]:
                    celda = ws.cell(
                        row=i + 2,
                        column=j + 2
                    )
                    celda.fill = PatternFill(
                        "solid",
                        fgColor="FFF2A8"
                    )

wb.save(archivo_excel)

# ============================================================
# 18. MOSTRAR RESULTADOS EN COLAB
# ============================================================

cartel(
    "ANÁLISIS FINALIZADO",
    f"""
    <b>Intervalo detectado:</b>
    {intervalo_texto(intervalo_detectado)}
    <br>
    <b>Muestras esperadas:</b>
    {total_esperados}
    <br>
    <b>Mediciones disponibles:</b>
    {total_medidos}
    <br>
    <b>Faltantes:</b>
    {total_faltantes}
    <br>
    <b>Cobertura:</b>
    {cobertura:.2f}%
    <br><br>
    <b>El heatmap ya no utiliza columnas
    de 20 minutos.</b>
    """,
    "#DCFCE7"
)

print("\n1. HEATMAP OPERATIVO")
fig_heatmap.show()

print("\n2. COMPORTAMIENTO DIARIO")
fig_diario.show()

print("\n3. COBERTURA DIARIA")
fig_cobertura.show()

print("\n4. ESTADISTICAS")
display(resumen.round(2))

# ============================================================
# 19. GENERAR HTML
# ============================================================

grafico_heatmap = plot(
    fig_heatmap,
    output_type="div",
    include_plotlyjs=True
)

grafico_diario = plot(
    fig_diario,
    output_type="div",
    include_plotlyjs=False
)

grafico_cobertura = plot(
    fig_cobertura,
    output_type="div",
    include_plotlyjs=False
)

tabla_resumen_html = resumen.round(2).to_html(
    index=False,
    classes="tabla",
    border=0,
    na_rep="-"
)

tabla_control_html = control_archivos.to_html(
    index=False,
    classes="tabla",
    border=0,
    na_rep="-"
)

tabla_muestreo_html = control_intervalos.to_html(
    index=False,
    classes="tabla",
    border=0,
    na_rep="-"
)

html_final = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport"
content="width=device-width,initial-scale=1">
<title>{html.escape(nombre)}</title>

<style>
body {{
    font-family:Arial,sans-serif;
    background:#f4f7fa;
    color:#263746;
    padding:25px;
}}
.contenedor {{
    max-width:1600px;
    margin:auto;
}}
h1 {{
    color:#315F85;
    text-align:center;
    font-size:30px;
}}
h2 {{
    color:#315F85;
}}
.panel {{
    background:white;
    padding:22px;
    margin:22px 0;
    border-radius:12px;
    box-shadow:0 3px 12px rgba(0,0,0,.08);
}}
.indicadores {{
    display:grid;
    grid-template-columns:
    repeat(auto-fit,minmax(190px,1fr));
    gap:14px;
}}
.indicador {{
    background:#e8f4fd;
    border-radius:10px;
    padding:20px;
    text-align:center;
}}
.valor {{
    font-size:27px;
    font-weight:bold;
    color:#315F85;
}}
.nombre {{
    font-size:14px;
    margin-top:8px;
}}
.tabla {{
    border-collapse:collapse;
    width:100%;
    font-size:13px;
}}
.tabla th {{
    background:#315F85;
    color:white;
    padding:10px;
    position:sticky;
    top:0;
}}
.tabla td {{
    padding:8px;
    text-align:center;
    border-bottom:1px solid #ddd;
}}
.tabla tr:nth-child(even) {{
    background:#f2f6fa;
}}
.tabla-contenedor {{
    overflow:auto;
    max-height:650px;
}}
.nota {{
    background:#fff3cd;
    border-left:5px solid #f59e0b;
    padding:17px;
    border-radius:8px;
    margin:20px 0;
}}
.grafico {{
    width:100%;
    overflow-x:auto;
}}
</style>
</head>

<body>
<div class="contenedor">

<h1>ANÁLISIS OPERATIVO DE CAUDAL</h1>

<div style="text-align:center;font-size:20px">
{html.escape(periodo)}
</div>

<div class="panel">
<h2>Indicadores Generales</h2>

<div class="indicadores">

<div class="indicador">
<div class="valor">
{intervalo_texto(intervalo_detectado)}
</div>
<div class="nombre">Muestreo detectado</div>
</div>

<div class="indicador">
<div class="valor">{len(dias_texto)}</div>
<div class="nombre">Días</div>
</div>

<div class="indicador">
<div class="valor">{len(serie)}</div>
<div class="nombre">Mediciones reales</div>
</div>

<div class="indicador">
<div class="valor">{total_faltantes}</div>
<div class="nombre">Posiciones faltantes</div>
</div>

<div class="indicador">
<div class="valor">{cobertura:.1f}%</div>
<div class="nombre">Cobertura</div>
</div>

<div class="indicador">
<div class="valor">{minimo_general:,.0f}</div>
<div class="nombre">Mínimo m³/h</div>
</div>

<div class="indicador">
<div class="valor">{promedio_general:,.0f}</div>
<div class="nombre">Promedio m³/h</div>
</div>

<div class="indicador">
<div class="valor">{maximo_general:,.0f}</div>
<div class="nombre">Máximo m³/h</div>
</div>

</div>
</div>

<div class="nota">

<b>CRITERIO DE RECONSTRUCCIÓN</b>

<br><br>

Intervalo detectado:
<b>{intervalo_texto(intervalo_detectado)}</b>.

<br><br>

El heatmap utiliza una columna por cada posición
de muestreo esperada dentro del día.

<br><br>

El eje X muestra las horas representativas
de esas posiciones. La hora exacta de cada
registro se muestra al colocar el cursor.

<br><br>

No se generan columnas fijas de 20 minutos.

<br><br>

No se agregan ceros artificiales.
No se interpolan valores.

<br><br>

<b>Cruz negra:</b> posición esperada sin medición.

<br>
<b>Cuadrado rojo:</b> medición bajo
{LIMITE_ANOMALIA} m³/h.

<br><br>

El gráfico diario conserva la banda celeste
mínimo-máximo y la línea azul de promedio.

</div>

<div class="panel">
<h2>1. Heatmap Operativo de Caudal</h2>
<div class="grafico">
{grafico_heatmap}
</div>
</div>

<div class="panel">
<h2>2. Comportamiento Diario</h2>
<div class="grafico">
{grafico_diario}
</div>
</div>

<div class="panel">
<h2>3. Cobertura Diaria de Mediciones</h2>
<div class="grafico">
{grafico_cobertura}
</div>
</div>

<div class="panel">
<h2>4. Tabla Resumen Diaria</h2>
<div class="tabla-contenedor">
{tabla_resumen_html}
</div>
</div>

<div class="panel">
<h2>5. Control de Archivos</h2>
<div class="tabla-contenedor">
{tabla_control_html}
</div>
</div>

<div class="panel">
<h2>6. Muestreo por Archivo</h2>
<div class="tabla-contenedor">
{tabla_muestreo_html}
</div>
</div>

</div>
</body>
</html>
"""

with open(
    archivo_html,
    "w",
    encoding="utf-8"
) as f:
    f.write(html_final)

# ============================================================
# 20. CONTROL FINAL
# ============================================================

print("\n" + "=" * 65)
print("CONTROL FINAL")
print("=" * 65)

print("Periodo:", periodo)

print(
    "Intervalo detectado:",
    intervalo_texto(intervalo_detectado)
)

print(
    "Intervalo medio:",
    f"{intervalo_reconstruccion:.2f} s"
)

print(
    "Columnas máximas del heatmap:",
    max_muestras_dia
)

print(
    "Posiciones esperadas:",
    total_esperados
)

print(
    "Posiciones medidas:",
    total_medidos
)

print(
    "Posiciones faltantes:",
    total_faltantes
)

print(
    "Mediciones no alineadas:",
    len(lecturas_no_alineadas)
)

print(f"Cobertura: {cobertura:.2f}%")
print("Promedio real:", promedio_general)

print("=" * 65)

# ============================================================
# 21. DESCARGAR
# ============================================================

cartel(
    "ARCHIVOS GENERADOS",
    f"""
    <b>{html.escape(archivo_excel)}</b>
    <br><br>
    <b>{html.escape(archivo_html)}</b>
    <br><br>
    Se iniciará la descarga automática.
    """,
    "#DCFCE7"
)

files.download(archivo_excel)
files.download(archivo_html)

print("\nPROCESO COMPLETADO")


Saving KAUDAL_AGUA RIO AGO26_2.csv to KAUDAL_AGUA RIO AGO26_2 (2).csv
Saving KAUDAL_AGUA RIO_AGO26-1.csv to KAUDAL_AGUA RIO_AGO26-1 (2).csv
KAUDAL_AGUA RIO AGO26_2 (2).csv: 770 registros válidos
KAUDAL_AGUA RIO_AGO26-1 (2).csv: 770 registros válidos


,Archivo,Intervalo (s),Intervalo detectado
0,KAUDAL_AGUA RIO_AGO26-1 (2).csv,1683.114434,28 min 3 s
1,KAUDAL_AGUA RIO AGO26_2 (2).csv,1683.114434,28 min 3 s



>>> NOMBRE DEL INFORME: REPORTE AGO2026



1. HEATMAP OPERATIVO



2. COMPORTAMIENTO DIARIO


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROCESO COMPLETADO
